# Arbitrage maladies Orphanet

Outil d'é**annotation manuelle** pour les patients dont le gène est associé à plusieurs maladies.

## Pipeline

```
convert_genes_to_diseases.ipynb
        ↓  patients_multi_diseases.csv
choice_disease.ipynb  (ce notebook)
        ↓  arbitrage_output.csv
        ↓  patients_arbitrage_final.csv
```

## Utilisation

1. Générer `patients_multi_diseases.csv` avec `convert_genes_to_diseases.ipynb`.
2. Exécuter les cellules **dans l'ordre**.
3. Dans l'interface : sélectionner la maladie la plus probable, puis **Confirmer**.
4. La progression est sauvegardée après **chaque** confirmation — vous pouvez fermer
   et reprendre sans perdre de travail.
5. Lancer la cellule **Export final** pour produire `patients_arbitrage_final.csv`.

## Fichiers

| Fichier | Rôle |
|---|---|
| `patients_multi_diseases.csv` | Entrée — généré par `convert_genes_to_diseases.ipynb` |
| `arbitrage_output.csv` | Annotations en cours (checkpoint) |
| `patients_arbitrage_final.csv` | Sortie finale (patients + annotations jointes) |


In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

In [ ]:
# ── Configuration ────────────────────────────────────────────
INPUT_FILE  = "patients_multi_diseases.csv"  # généré par convert_genes_to_diseases.ipynb
OUTPUT_FILE = "arbitrage_output.csv"

df = pd.read_csv(INPUT_FILE)
all_ids = list(df["ID_PAT_ETUDE"].unique())

# Reprise de session : on relit le fichier de sortie s'il existe déjà,
# pour sauter les patients déjà annotés lors d'une exécution précédente.
try:
    done_ids = set(pd.read_csv(OUTPUT_FILE)["ID_PAT_ETUDE"].unique())
except FileNotFoundError:
    done_ids = set()

remaining = [p for p in all_ids if p not in done_ids]

print(f"Total patients : {len(all_ids)}")
print(f"Déjà annotés   : {len(done_ids)}")
print(f"Restants       : {len(remaining)}")
if done_ids:
    print(f"↩  Reprise au patient {len(done_ids) + 1}")


## Interface d'arbitrage

Lancez la cellule ci-dessous. Pour chaque patient :

- Les **termes HPO** du patient sont affichés pour guider le choix clinique.
- Sélectionnez la maladie la plus probable parmi les options proposées.
- Cliquez sur **✔ Confirmer** pour enregistrer et passer au suivant.
- **Passer →** saute le patient sans l'annoter (il restera dans `remaining` à la prochaine session).

> La progression est sauvegardée sur disque après chaque confirmation.
> Vous pouvez interrompre et reprendre à tout moment.


In [ ]:
# `state` est un dict (pas un int) afin que les callbacks on_confirm/on_skip
# puissent modifier l'index via state["idx"] += 1 — une variable locale
# reboundée dans une closure ne remonterait pas au scope appelant.
state = {"idx": 0}


def save_result(pat_id, disease_name, orpha_code):
    """Persiste l'annotation d'un patient dans OUTPUT_FILE.

    On relit le CSV à chaque appel plutôt que de travailler en mémoire :
    cela garantit la cohérence si le kernel est relancé en cours de session.
    """
    new_row = pd.DataFrame([{
        "ID_PAT_ETUDE"     : pat_id,
        "real_disease_name": disease_name,
        "real_disease_code": orpha_code,
    }])
    try:
        current = pd.read_csv(OUTPUT_FILE)
        updated = pd.concat([current, new_row]).drop_duplicates("ID_PAT_ETUDE", keep="last")
    except FileNotFoundError:
        updated = new_row
    updated.to_csv(OUTPUT_FILE, index=False)


# `banner` est défini hors de `render` pour survivre aux appels successifs :
# clear_output() efface les widgets créés dans `out`, mais pas `banner` lui-même,
# ce qui permet d'afficher un bandeau de confirmation persistant entre patients.
banner = widgets.HTML(value="")


def render(idx, last_saved=None):
    if last_saved:
        banner.value = (
            f'<div style="color:#2e7d32; background:#e8f5e9; border-radius:6px; '
            f'padding:6px 12px; margin-bottom:6px; font-family:monospace">'
            f'✓ Sauvegardé : {last_saved}</div>'
        )

    if idx >= len(remaining):
        with out:
            clear_output(wait=True)
            display(banner)
            display(widgets.HTML(
                '<div style="color:#1565c0; font-size:15px; padding:12px">'
                '✅ Tous les patients ont été annotés !<br>'
                f'Fichier : <b>{OUTPUT_FILE}</b></div>'
            ))
        return

    pat_id   = remaining[idx]
    pat_data = df[df["ID_PAT_ETUDE"] == pat_id]

    dossier = (
        pat_data["nom_dossier"].iloc[0]
        if "nom_dossier" in pat_data.columns and pd.notna(pat_data["nom_dossier"].iloc[0])
        else "—"
    )
    hpo = (
        pat_data["hpo_terms"].iloc[0]
        if "hpo_terms" in pat_data.columns and pd.notna(pat_data["hpo_terms"].iloc[0])
        else "—"
    )
    gene = ", ".join(pat_data["gene_symbol"].unique())

    progress = widgets.IntProgress(
        value=idx, min=0, max=len(remaining),
        description=f"{idx}/{len(remaining)}",
        bar_style="info",
        layout=widgets.Layout(width="100%"),
    )

    header = widgets.HTML(f"""
        <div style=\"border:1px solid #d0d0d0; border-radius:8px; padding:16px;
                    background:#f8f8f8; font-family:monospace; font-size:13px; margin-bottom:8px\">
          <b style=\"font-size:14px\">Patient {idx + 1} / {len(remaining)}</b><br><br>
          <b>Dossier&nbsp;&nbsp;&nbsp;:</b> {dossier}<br>
          <b>ID patient :</b> {pat_id}<br>
          <b>Gène&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;:</b> {gene}<br>
          <br>
          <b>Termes HPO :</b><br>
          <span style=\"color:#444; white-space:pre-wrap\">{hpo}</span>
        </div>
    """)

    choices = pat_data[["OrphaCode", "DiseaseName"]].drop_duplicates()
    # Chaque option est un tuple (label affiché, OrphaCode, DiseaseName)
    options = [
        (f"{row['DiseaseName']}  (ORPHA:{row['OrphaCode']})", row["OrphaCode"], row["DiseaseName"])
        for _, row in choices.iterrows()
    ]

    radio = widgets.RadioButtons(
        options=[o[0] for o in options],
        layout=widgets.Layout(width="100%"),
    )

    btn_confirm = widgets.Button(
        description="✔ Confirmer",
        button_style="success",
        layout=widgets.Layout(width="160px"),
    )
    btn_skip = widgets.Button(
        description="Passer →",
        button_style="warning",
        layout=widgets.Layout(width="110px"),
    )

    def on_confirm(b):
        # Désactivation immédiate pour éviter les double-clics
        btn_confirm.disabled = True
        btn_skip.disabled    = True
        sel = next(o for o in options if o[0] == radio.value)
        save_result(pat_id, sel[2], sel[1])
        state["idx"] += 1
        render(state["idx"], last_saved=sel[2])

    def on_skip(b):
        state["idx"] += 1
        render(state["idx"], last_saved=None)

    btn_confirm.on_click(on_confirm)
    btn_skip.on_click(on_skip)

    ui = widgets.VBox([
        progress,
        header,
        widgets.HTML("<b>Sélectionnez la maladie :</b>"),
        radio,
        widgets.HBox([btn_confirm, btn_skip]),
    ])

    with out:
        clear_output(wait=True)
        display(banner)
        display(ui)


out = widgets.Output()
display(out)
render(0)


## Export final

Relancez la cellule ci-dessous **après avoir terminé toutes les annotations**.

Elle joint `arbitrage_output.csv` (annotations) au fichier source pour produire
`patients_arbitrage_final.csv` : une ligne par patient avec `real_disease_name`
et `real_disease_code` renseignés.

> Les colonnes `OrphaCode`, `DiseaseName`, `GeneSymbol` et `GeneId` (multivaluées)
> sont retirées de l'export final pour éviter les doublons de lignes.


In [ ]:
annotations = pd.read_csv(OUTPUT_FILE)

# On déduplicate sur ID_PAT_ETUDE avant le merge pour obtenir une ligne par patient :
# le CSV source contient plusieurs lignes par patient (une par maladie candidate).
df_patients = df.drop_duplicates("ID_PAT_ETUDE")[
    [c for c in df.columns if c not in ["OrphaCode", "DiseaseName", "GeneSymbol", "GeneId"]]
]

df_export = df_patients.merge(annotations, on="ID_PAT_ETUDE", how="left")
df_export.to_csv("patients_arbitrage_final.csv", index=False)

annotated = df_export["real_disease_code"].notna().sum()
print(f"Export : patients_arbitrage_final.csv")
print(f"Annotés   : {annotated} / {len(df_export)}")
df_export.head()
